# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a Croissant-based FAIR dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print("Dataset Title: {}".format(dataset.metadata.name))
print("Description: {}".format(dataset.metadata.description))

## 2. Data Overview
Review available record sets and their fields, including all `@id` values for reference.

In [ ]:
# List all record sets and their field @ids
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}, @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field name: {field.name}, @id: {field.id}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use `@id` values.

In [ ]:
# ========== CUSTOMIZATION =======================
# If record sets were found above, enter their `@id` values here.
# If none found, try to guess typical Croissant datasets main record set ids like 'ProteinRecordSet'.
# ===============================================
#
# If the above cell did not yield results, supply candidate record set @id from Croissant conventions:
record_set_ids = []
# Try to probe all known record_set @ids
if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
else:
    # Fallback to conventions (update as needed, user may update these once they explore the dataset)
    record_set_ids = ['cr:ProteinRecordSet', 'cr:records']  # Try guessing standard names

dataframes = {}
for record_set_id in record_set_ids:
    try:
        print(f"Attempting to load records from record set '@id': {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for {record_set_id}, columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record set '@id': {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Pick the first available record set as our working data
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Using record set '@id': {main_record_set_id} for analysis.")
    print("Sample columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded. Please check the record set @ids and try again.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

All columns and group-by fields are referenced by their `@id`.

**Note:** Edit `numeric_field_id` and `group_field_id` as needed based on your schema's field `@id` and DataFrame columns.

In [ ]:
import numpy as np
if dataframes:
    df = dataframes[main_record_set_id]
    # Try to auto-select a numeric field, else user can edit this cell
    # As an example, try common names found in mass spec data
    candidate_numeric_fields = ['cr:MW', 'cr:Abundance', 'cr:PeptideCount', 'cr:Coverage']
    numeric_field_id = None
    for nf in candidate_numeric_fields:
        if nf in df.columns:
            numeric_field_id = nf
            break
    if not numeric_field_id:
        # If not found, default to the first numeric column
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field_id = col
                break
    if numeric_field_id:
        print(f"Selecting numeric field for EDA: {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 1
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df.loc[:, f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a categorical field. Edit as needed.
        candidate_group_fields = ['cr:Description', 'cr:ProteinClass', 'cr:SampleType']
        group_field_id = None
        for gf in candidate_group_fields:
            if gf in df.columns:
                group_field_id = gf
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}, displaying mean {numeric_field_id} per group:")
            display(grouped_df.head())
        else:
            print("No appropriate group field found for grouping. You may edit 'candidate_group_fields' to match your schema.")
    else:
        print("No numeric field found in this record set. Please edit the variable 'numeric_field_id' above.")
else:
    print("No DataFrame loaded for EDA. Please check previous steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if dataframes and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If normalized field was computed
    if f"{numeric_field_id}_normalized" in df.columns:
        plt.figure(figsize=(8,5))
        sns.histplot(df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of normalized {numeric_field_id}")
        plt.xlabel(f"Normalized {numeric_field_id}")
        plt.ylabel("Count")
        plt.show()
else:
    print("No numeric field available for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


* In this notebook, we demonstrated loading and exploring a mass spectrometry-based proteomics dataset described by a Croissant schema (`mlcroissant`).
* The records and columns are referenced exclusively using their Croissant `@id` fields, ensuring robust querying and analysis.
* The notebook allows you to:
  - Inspect available record sets and their field definitions
  - Load record data into pandas DataFrames by record set `@id`
  - Perform basic EDA such as filtering, normalization, and grouping using both numeric and categorical field `@id`s
  - Visualize data distributions for selected numeric fields
* For advanced use—such as joining multiple record sets, deeper feature analysis, or machine learning—adapt this template to leverage the `mlcroissant` library and the Croissant schema's semantic richness.

👉 **Edit the candidate field lists or supply actual `@id`s as discovered via Section 2 to customize this notebook for the FAIR² dataset or your Croissant dataset!**